# 34 - Build a pooled, trust-checked evaluation set

Since there is no gold standard (confirmed by Robert) and the frozen `goi_search_results.json` is just one system's output, this builds a pooled candidate set per query, TREC-style, from every retrieval system already run: the frozen ground truth, a fresh v2 API pull (where available), BM25, GTE-large, ColBERT, and the learned fusion ranker.

Before any relevance judging happens (that's the next notebook), every pooled candidate's summary is run through the existing boilerplate/contamination detector from `trust_feature.py`. A candidate whose AI-generated summary describes the wrong entity (e.g. a hosting provider instead of the real company) would poison any relevance judgment built on top of it, so this flags that up front rather than silently baking bad labels into the new evaluation set.

Output: `result/34_pooled_evaluation_set/pooled_candidates.json`, one row per (query, candidate), with which source(s) retrieved it and its trust flag.

In [1]:
import json
import pandas as pd
from pathlib import Path

TOP_N = 20  # candidates pulled per source per query, before dedup
OUTPUT_DIR = Path("result/34_pooled_evaluation_set")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

corpus = pd.read_csv("dataset/company_corpus.csv")
QUERY_IDS = sorted(corpus["query_id"].unique().tolist())
print(f"Queries: {len(QUERY_IDS)}")

Queries: 101


In [2]:
# Global domain -> text lookup. Every retrieval method searches the same local
# corpus, so any domain any of them returns is guaranteed to be in here already.
lookup = corpus.drop_duplicates(subset="domain").set_index("domain")


def lookup_text(domain):
    if domain not in lookup.index:
        return None
    row = lookup.loc[domain]
    return {
        "name": row["name"],
        "summary": row["summary"],
        "summary_keywords": row["summary_keywords"],
        "country": row["country"],
    }

In [3]:
def top_n_by_rank(df, query_id, n, rank_col="rank"):
    sub = df[df["query_id"] == query_id].sort_values(rank_col)
    return sub["domain"].head(n).tolist()


# ---- Ground truth (frozen production output) ----
ground_truth = corpus[["query_id", "domain", "rank"]]

# ---- BM25 ----
bm25 = pd.read_csv("result/01_baseline_bm25/bm25_results.csv")[["query_id", "domain", "rank"]]

# ---- GTE-large (best solo embedding baseline) ----
gte = pd.read_csv("result/17_baseline_gte_large/gte_large_results.csv")[["query_id", "domain", "rank"]]

# ---- ColBERT ----
colbert = pd.read_csv("result/27_baseline_colbert/colbert_results.csv")[["query_id", "domain", "rank"]]

# ---- Learned fusion ranker: no rank column, derive it from score_gbdt ----
fusion = pd.read_csv("result/30_learned_fusion_ranker/scored_candidates.csv")
fusion["rank"] = fusion.groupby("query_id")["score_gbdt"].rank(ascending=False, method="first")
fusion = fusion[["query_id", "domain", "rank"]]

# ---- Fresh v2 API pulls, wherever they already exist (optional -- notebooks 32/33) ----
fresh_api_rows = []
single_query_path = Path("result/32_api_v2_check/software_companies_v2_1000.json")
if single_query_path.exists():
    rows = json.load(open(single_query_path))
    for row in rows:
        fresh_api_rows.append({"query_id": 1, "domain": row["domain"], "rank": row["rank"]})

oversample_cache_path = Path("result/33_api_v2_oversample/oversample_cache.json")
if oversample_cache_path.exists():
    cache = json.load(open(oversample_cache_path))
    for qid_str, entry in cache.items():
        for row in entry["results"]:
            fresh_api_rows.append({"query_id": int(qid_str), "domain": row["domain"], "rank": row["rank"]})

fresh_api = pd.DataFrame(fresh_api_rows) if fresh_api_rows else pd.DataFrame(columns=["query_id", "domain", "rank"])
print(f"Fresh API coverage: {fresh_api['query_id'].nunique() if len(fresh_api) else 0}/{len(QUERY_IDS)} queries")

SOURCES = {
    "ground_truth": ground_truth,
    "bm25": bm25,
    "gte_large": gte,
    "colbert": colbert,
    "learned_fusion": fusion,
    "fresh_api": fresh_api,
}
for name, df in SOURCES.items():
    print(f"  {name}: {len(df)} rows, {df['query_id'].nunique()} queries")

Fresh API coverage: 2/101 queries
  ground_truth: 98716 rows, 101 queries
  bm25: 101000 rows, 101 queries
  gte_large: 101000 rows, 101 queries
  colbert: 101000 rows, 101 queries
  learned_fusion: 173262 rows, 101 queries
  fresh_api: 3500 rows, 2 queries


In [4]:
pooled_rows = []

for query_id in QUERY_IDS:
    query_text = corpus[corpus["query_id"] == query_id]["query"].iloc[0]
    domain_sources = {}  # domain -> list of contributing source names

    for source_name, df in SOURCES.items():
        if query_id not in df["query_id"].values:
            continue
        for domain in top_n_by_rank(df, query_id, TOP_N):
            domain_sources.setdefault(domain, []).append(source_name)

    for domain, sources in domain_sources.items():
        text = lookup_text(domain)
        if text is None:
            continue  # shouldn't happen -- every domain comes from the same shared corpus
        pooled_rows.append({
            "query_id": query_id, "query": query_text, "domain": domain,
            "sources": sources, "n_sources": len(sources), **text,
        })

pooled_df = pd.DataFrame(pooled_rows)
print(f"Pooled candidates: {len(pooled_df)} rows across {pooled_df['query_id'].nunique()} queries")
print(f"Average unique candidates per query: {len(pooled_df) / pooled_df['query_id'].nunique():.1f}")

Pooled candidates: 8309 rows across 101 queries
Average unique candidates per query: 82.3


In [5]:
# Trust check -- flag candidates whose AI-generated summary looks like it describes
# the wrong entity (hosting provider, registrar, etc.) BEFORE any relevance judging.
import trust_feature as tf

embedder = tf.get_embedder()
template_texts, template_embeddings = tf.load_boilerplate_templates(embedder=embedder)
print(f"Loaded {len(template_texts)} boilerplate templates")

verdicts = tf.summary_verdicts_batch(
    pooled_df["name"].tolist(),
    pooled_df["domain"].tolist(),
    pooled_df["summary"].tolist(),
    template_embeddings,
    embedder,
)
pooled_df["summary_trustworthy"] = [ok for ok, _ in verdicts]
pooled_df["summary_reasons"] = ["; ".join(reasons) for _, reasons in verdicts]

n_bad = (~pooled_df["summary_trustworthy"]).sum()
print(f"Flagged untrustworthy: {n_bad}/{len(pooled_df)} ({100*n_bad/len(pooled_df):.1f}%)")

/home/ma/ma_ma/ma_mpandya/Thesis/thesis/lib64/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1834.35it/s]


Loaded 296 boilerplate templates
Flagged untrustworthy: 487/8309 (5.9%)


In [6]:
out_path = OUTPUT_DIR / "pooled_candidates.json"
pooled_df.to_json(out_path, orient="records", indent=2)
print(f"Saved -> {out_path}")
print()
print("Per-query pool size distribution:")
print(pooled_df.groupby("query_id").size().describe())
print()
print("Contamination rate by number of contributing sources (more sources agreeing may correlate with trust):")
print(pooled_df.groupby("n_sources")["summary_trustworthy"].mean())

Saved -> result/34_pooled_evaluation_set/pooled_candidates.json

Per-query pool size distribution:
count    101.000000
mean      82.267327
std        7.303275
min       58.000000
25%       79.000000
50%       83.000000
75%       87.000000
max       98.000000
dtype: float64

Contamination rate by number of contributing sources (more sources agreeing may correlate with trust):
n_sources
1    0.935663
2    0.964897
3    0.983740
4    0.962264
5    1.000000
Name: summary_trustworthy, dtype: float64
